# 北上广深租房市场数据分析

## 第三部分：数据清洗

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/data_sample.csv")

df = pd.read_csv(DATA_PATH)

clean_df = df.copy()

print("原始数据：", df.shape)
print("清洗副本：", clean_df.shape)

原始数据： (12000, 20)
清洗副本： (12000, 20)


## 一、清洗面积和租金字段

In [3]:
# 再次确认原始字段，确保它们和Day2的分析结果一致
clean_df[["rent_area", "rent_price_listing"]].head()

,rent_area,rent_price_listing
0,137,15000
1,57,4500
2,56,10500
3,43,5600
4,56,6300


In [4]:
# 数一数面积和租金中有多少个区间值
area_range_count = clean_df["rent_area"].astype(str).str.contains("-").sum()

price_range_count = clean_df["rent_price_listing"].astype(str).str.contains("-").sum()

print("面积区间数量：", area_range_count)
print("租金区间数量：", price_range_count)
# 与Day2一致

面积区间数量： 250
租金区间数量： 524


In [5]:
# 把面积和租金转换成可以计算的数字（范围中间值）。
def convert_range(value):
    text = str(value).strip()

    if "-" in text:
        numbers = text.split("-")
        low = float(numbers[0])
        high = float(numbers[1])
        return (low + high) / 2

    return float(text)

print(convert_range("30-35"))
print(convert_range("60"))
print(convert_range("1400-1500"))

32.5
60.0
1450.0


In [7]:
# 把刚才创建的转换工具，真正应用到全部12,000条数据上
clean_df["rent_area_num"] = clean_df["rent_area"].apply(convert_range) # .apply：把一个函数依次应用到一列中的每个值上

clean_df["rent_price_num"] = clean_df["rent_price_listing"].apply(convert_range)

清洗后的数据规模： (12000, 22)


In [8]:
# 检查新字段是否创建成功，以及有没有转换失败
print("数据规模：", clean_df.shape)

print("面积字段类型：", clean_df["rent_area_num"].dtype)
print("租金字段类型：", clean_df["rent_price_num"].dtype)

print("面积转换失败数量：", clean_df["rent_area_num"].isna().sum())
print("租金转换失败数量：", clean_df["rent_price_num"].isna().sum())

数据规模： (12000, 22)
面积字段类型： float64
租金字段类型： float64
面积转换失败数量： 0
租金转换失败数量： 0


## 二、处理缺失值

In [12]:
# 把卧室数和卫生间数中的 0 改成缺失值
clean_df["bedroom_num"] = clean_df["bedroom_num"].replace(0, np.nan)

clean_df["bathroom_num"] = clean_df["bathroom_num"].replace(0, np.nan)

In [13]:
# 处理文字字段中的缺失值
clean_df["house_tag"] = clean_df["house_tag"].fillna("无标签")

clean_df["resblock_name"] = clean_df["resblock_name"].fillna("未知小区")

clean_df["frame_orientation"] = clean_df["frame_orientation"].fillna("未知朝向")

clean_df["bizcircle_name"] = clean_df["bizcircle_name"].fillna("未知商圈")

In [14]:
# 建立一个需要检查的字段清单。它不会修改数据，只是把多个字段名放在一起，方便后面统一检查
check_columns = [
    "bedroom_num",
    "bathroom_num",
    "house_tag",
    "resblock_name",
    "frame_orientation",
    "bizcircle_name",
    "distance",
    "latitude",
    "longitude"
]

In [15]:
# 统计这些字段还剩多少缺失值
clean_df[check_columns].isna().sum()

bedroom_num             3
bathroom_num           87
house_tag               0
resblock_name           0
frame_orientation       0
bizcircle_name          0
distance             5206
latitude               31
longitude              31
dtype: int64

## 三、识别并排除非住宅房源

In [17]:
# 找出“混进租房数据里的非住宅”（因为统计的是住宅）
def check_non_residential(row):
    title = str(row["house_title"])
    bathroom = row["bathroom_num"]

    if "仓库" in title:
        return True

    if "底层商铺" in title or "临街商铺" in title:
        return True

    if "不能住人" in title or "地下停车场" in title:
        return True

    if "只出租一个产权车位" in title or "区车位，不限行" in title:
        return True

    if pd.isna(bathroom):
        if (
            "写字楼" in title
            or "办公楼" in title
            or "商办类" in title
            or "办公神器" in title
            or "用来办公，研发" in title
        ):
            return True

    return False
# 标题包含“仓库”          → 非住宅，返回True
# 标题明确写着“商铺”      → 非住宅，返回True
# 标题明确表示停车场或车位 → 非住宅，返回True
# 卫生间未知并明确是办公房 → 非住宅，返回True
# 以上条件都不符合         → 返回False
# 为什么办公房还要检查卫生间？因为：三居室，可居家可办公可能仍然是住宅，不能看到“办公”就删除。
# 但是：写字楼 + 卫生间数量未知更可能属于非住宅。

In [20]:
# 创建函数后，用下面的代码处理每一行
clean_df["is_non_residential"] = clean_df.apply(
    check_non_residential,
    axis=1
)

# 最后统计数量
print(
    "非住宅记录数量：",
    clean_df["is_non_residential"].sum()
)

非住宅记录数量： 22


In [25]:
# 把刚才标记的22条非住宅记录，从清洗数据中排除
clean_df = clean_df[clean_df["is_non_residential"] == False].copy()

clean_df = clean_df.drop(
    columns=["is_non_residential"]  # 这列之前用于判断哪些记录需要排除。非住宅记录删除以后，剩下的记录全部都是 False，所以这列已经没有用了。
)

clean_df = clean_df.reset_index(  # 重新整理行号
    drop=True
)

## 四、添加异常记录标记

In [26]:
# 标记可疑记录，但不删除它们。因为小面积或低租金不一定是错误数据，后面可以根据分析目的决定是否排除。
clean_df["is_small_entire"] = (
    (clean_df["type"] == "整租")
    & (clean_df["rent_area_num"] <= 10)
)

clean_df["is_large_area_conflict"] = (
    (clean_df["rent_area_num"] >= 1000)
    & (clean_df["bedroom_num"] <= 1)
    & (clean_df["rent_price_num"] <= 5000)
)

clean_df["is_low_price_entire"] = (
    (clean_df["type"] == "整租")
    & (clean_df["rent_price_num"] <= 500)
)

# 最后统一统计一次
anomaly_columns = [
    "is_small_entire",
    "is_large_area_conflict",
    "is_low_price_entire"
]

clean_df[anomaly_columns].sum()
# 这些记录仍然保留在 clean_df 中，只是增加了 True 异常标记

is_small_entire           11
is_large_area_conflict     1
is_low_price_entire       27
dtype: int64

## 五、计算每平方米租金

In [27]:
# 为了让不同面积的房源可以公平比较
clean_df["rent_price_per_sqm"] = (
    clean_df["rent_price_num"]
    / clean_df["rent_area_num"]
).round(2)  # 把结果保留两位小数

# 最后查看几条计算结果
clean_df[
    [
        "rent_price_num",
        "rent_area_num",
        "rent_price_per_sqm"
    ]
].head()

,rent_price_num,rent_area_num,rent_price_per_sqm
0,15000.0,137.0,109.49
1,4500.0,57.0,78.95
2,10500.0,56.0,187.50
3,5600.0,43.0,130.23
4,6300.0,56.0,112.50


## 六、检查清洗结果

In [28]:
# Day3的统一质量检查，确认：
# 非住宅记录已经排除。
# 数值字段转换成功。
# 没有产生重复记录。
# 异常标记数量正确。
# 已填充的分类字段不再缺失。
print("原始数据规模：", df.shape)
print("清洗数据规模：", clean_df.shape)

print("重复ID数量：", clean_df["_id"].duplicated().sum())

print("面积数值缺失：", clean_df["rent_area_num"].isna().sum())
print("租金数值缺失：", clean_df["rent_price_num"].isna().sum())
print("每平方米租金缺失：", clean_df["rent_price_per_sqm"].isna().sum())

category_columns = [
    "house_tag",
    "resblock_name",
    "frame_orientation",
    "bizcircle_name"
]

print(
    "已处理分类字段的缺失总数：",
    clean_df[category_columns].isna().sum().sum()
)

print("小面积整租标记：", clean_df["is_small_entire"].sum())
print("超大面积冲突标记：", clean_df["is_large_area_conflict"].sum())
print("低价整租标记：", clean_df["is_low_price_entire"].sum())


原始数据规模： (12000, 20)
清洗数据规模： (11978, 26)
重复ID数量： 0
面积数值缺失： 0
租金数值缺失： 0
每平方米租金缺失： 0
已处理分类字段的缺失总数： 0
小面积整租标记： 11
超大面积冲突标记： 1
低价整租标记： 27


In [29]:
# 确认实际多了哪些列
for column in clean_df.columns:
    if column not in df.columns:
        print(column)

rent_area_num
rent_price_num
is_small_entire
is_large_area_conflict
is_low_price_entire
rent_price_per_sqm


## 七、保存清洗后的数据

In [31]:
# 把内存中的 clean_df 保存为新的 CSV 文件。原始文件不会被覆盖\
OUTPUT_PATH = Path(
    "../data/processed/rent_cleaned.csv"
)

clean_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# 保存后统一验证一次
saved_df = pd.read_csv(OUTPUT_PATH)

print("保存位置：", OUTPUT_PATH)
print("保存后的数据规模：", saved_df.shape)

保存位置： ..\data\processed\rent_cleaned.csv
保存后的数据规模： (11978, 26)


## 数据清洗总结

### 数据规模变化

- 原始数据包含12,000条租房记录和20个字段。
- 排除22条明确属于写字楼、办公房、车位、商铺或仓库的非住宅记录。
- 清洗后保留11,978条住宅租房记录。
- 清洗后共有26个字段。
- 原始房源ID没有重复，清洗过程没有产生新的重复记录。

### 面积和租金清洗

- 保留原始面积字段 `rent_area` 和原始租金字段 `rent_price_listing`。
- 新增数值面积字段 `rent_area_num`。
- 新增数值租金字段 `rent_price_num`。
- 250条面积区间统一转换为区间中点。
- 524条租金区间统一转换为区间中点。
- 面积和租金全部转换成功，没有产生新的缺失值。
- 新增每平方米租金字段 `rent_price_per_sqm`，用于比较不同面积房源的租金水平。

### 缺失值处理

- `bedroom_num`为0的记录改为缺失值，不把未知卧室数量当作真实的0。
- `bathroom_num`为0的记录改为缺失值。
- `house_tag`缺失值填充为“无标签”。
- `resblock_name`缺失值填充为“未知小区”。
- `frame_orientation`缺失值填充为“未知朝向”。
- `bizcircle_name`缺失值填充为“未知商圈”。
- `distance`缺失值不填平均数，只在地铁距离分析时排除。
- 经纬度缺失值不进行人工填补，只在地图分析时排除。

### 非住宅记录处理

- 根据房源标题和卫生间信息识别明确的非住宅记录。
- 写字楼、办公楼、商办房、停车位、商铺和仓库等记录被排除。
- 没有因为标题中普通出现“办公”或“车位”就直接删除记录。
- 原始CSV和原始数据变量 `df` 始终保持不变。

### 异常记录标记

- 11条面积不超过10平方米的整租房源添加小面积整租标记。
- 1条面积、户型和租金明显冲突的记录添加超大面积冲突标记。
- 27条月租不超过500元的整租房源添加低价整租标记。
- 异常记录只添加标记，没有直接删除。
- 高卧室数、高卫生间数和高租金记录如果与户型及面积相符，则继续保留。

### 清洗结果

- 清洗结果保存到 `data/processed/rent_cleaned.csv`。
- 最终数据规模为11,978行和26列。
- 清洗后的数据可以用于后续城市比较、行政区分析、每平方米租金分析、SQL查询和数据可视化。
- 后续分析中仍需将整租和合租分开，避免面积与户型含义不同造成误导。